<div>
<img src=https://www.institutedata.com/wp-content/uploads/2019/10/iod_h_tp_primary_c.svg width="300">
</div>

# Lab 3.2.1
# *Querying the International Space Station*

## The OpenNotify API

The OpenNotify API exposes a few attributes of the International Space Station (ISS) via a simple, authentication-free interface. The simplicity of this API precludes any need for a dedicated Python library. However, as with many APIs, it accepts requests according to HTTP standards and returns responses in JSON format, so the Python libraries request and json will make managing the I/O simpler still.

In [1]:
import requests
import json
from datetime import datetime, date, time

This request fetches the latest position of the international space station:

In [2]:
response = requests.get("http://api.open-notify.org/iss-now.json")

Print the status code and text of the response:

In [3]:
#ANSWER
print(response.status_code)

200


In [4]:
#ANSWER
print(response.text)

{"iss_position": {"latitude": "0.9639", "longitude": "173.9496"}, "timestamp": 1774397040, "message": "success"}


We can use another API to request the current position of the ISS and the next few times at which it will be over a certain location. The latitude and longitude of Sydney are (-33.87, 151.21).

In [5]:
response = requests.get("https://api.g7vrd.co.uk/v1/satellite-passes/25544/-33.87/151.21.json?minelevation=0&hours=24")

Print the response header:

In [6]:
#ANSWER
print(response.headers)

{'Date': 'Wed, 25 Mar 2026 00:04:00 GMT', 'Server': 'Apache', 'X-Rate-Limit-Remaining': '9', 'Vary': 'Origin,Access-Control-Request-Method,Access-Control-Request-Headers', 'Access-Control-Allow-Origin': '*', 'X-Content-Type-Options': 'nosniff', 'X-XSS-Protection': '0', 'Cache-Control': 'no-cache, no-store, max-age=0, must-revalidate', 'Pragma': 'no-cache', 'Expires': '0', 'X-Frame-Options': 'DENY', 'Content-Type': 'application/json', 'Keep-Alive': 'timeout=5, max=100', 'Connection': 'Keep-Alive', 'Transfer-Encoding': 'chunked'}


Print the content of the response (the data that the server returned):

In [7]:
#ANSWER
print(response.content)

b'{"api_status":"ALPHA","request_timestamp":"2026-03-25T00:04:00.965522189Z","norad_id":25544,"satellite_name":"ISS","tle_last_retrieved":"2026-03-24T22:41:35.248538153Z","lat":-33.87,"lon":151.21,"hours":24,"min_elevation":0,"query_ms":14,"passes":[{"start":"2026-03-25T01:28:00.952Z","tca":"2026-03-25T01:29:00.952Z","end":"2026-03-25T01:30:25.952Z","aos_azimuth":288,"los_azimuth":313,"max_elevation":0.0},{"start":"2026-03-25T13:19:50.952Z","tca":"2026-03-25T13:20:50.952Z","end":"2026-03-25T13:22:20.952Z","aos_azimuth":46,"los_azimuth":73,"max_elevation":1.0},{"start":"2026-03-25T14:51:45.952Z","tca":"2026-03-25T14:57:15.952Z","end":"2026-03-25T15:02:30.952Z","aos_azimuth":329,"los_azimuth":126,"max_elevation":41.0},{"start":"2026-03-25T16:29:05.952Z","tca":"2026-03-25T16:34:05.952Z","end":"2026-03-25T16:39:10.952Z","aos_azimuth":278,"los_azimuth":145,"max_elevation":19.0},{"start":"2026-03-25T18:08:40.952Z","tca":"2026-03-25T18:12:10.952Z","end":"2026-03-25T18:15:30.952Z","aos_azimuth

Note that this is a Python byte string:

In [8]:
print(type(response.content))

<class 'bytes'>


Print just the "content-type" value from the header:

In [9]:
#ANSWER
print(response.headers['content-type'])

application/json


JSON was designed to be easy for computers to read, not for people. The `requests` library can decode the JSON byte string:

In [10]:
overheads = response.json()
print(overheads)

{'api_status': 'ALPHA', 'request_timestamp': '2026-03-25T00:04:00.965522189Z', 'norad_id': 25544, 'satellite_name': 'ISS', 'tle_last_retrieved': '2026-03-24T22:41:35.248538153Z', 'lat': -33.87, 'lon': 151.21, 'hours': 24, 'min_elevation': 0, 'query_ms': 14, 'passes': [{'start': '2026-03-25T01:28:00.952Z', 'tca': '2026-03-25T01:29:00.952Z', 'end': '2026-03-25T01:30:25.952Z', 'aos_azimuth': 288, 'los_azimuth': 313, 'max_elevation': 0.0}, {'start': '2026-03-25T13:19:50.952Z', 'tca': '2026-03-25T13:20:50.952Z', 'end': '2026-03-25T13:22:20.952Z', 'aos_azimuth': 46, 'los_azimuth': 73, 'max_elevation': 1.0}, {'start': '2026-03-25T14:51:45.952Z', 'tca': '2026-03-25T14:57:15.952Z', 'end': '2026-03-25T15:02:30.952Z', 'aos_azimuth': 329, 'los_azimuth': 126, 'max_elevation': 41.0}, {'start': '2026-03-25T16:29:05.952Z', 'tca': '2026-03-25T16:34:05.952Z', 'end': '2026-03-25T16:39:10.952Z', 'aos_azimuth': 278, 'los_azimuth': 145, 'max_elevation': 19.0}, {'start': '2026-03-25T18:08:40.952Z', 'tca': '2

What kind of object did this give us?

In [11]:
#ANSWER:
print(type(overheads))

<class 'dict'>


Python dicts are easier to work with, but the data we want is still buried in that data structure, so we have to dig it out. First, extract the `passes` value to a separate list:

In [12]:
#ANSWER:
passes = overheads['passes']

Now extract the `start` strings into an array called `srisetimes`:

In [13]:
#ANSWER:
srisetimes = [xpass['start'] for xpass in passes]
srisetimes

['2026-03-25T01:28:00.952Z',
 '2026-03-25T13:19:50.952Z',
 '2026-03-25T14:51:45.952Z',
 '2026-03-25T16:29:05.952Z',
 '2026-03-25T18:08:40.952Z',
 '2026-03-25T19:47:40.952Z',
 '2026-03-25T21:24:10.952Z',
 '2026-03-25T23:00:40.952Z',
 '2026-03-26T00:38:55.952Z']

These are strings. We convert these to an array of Python `datetime` values called `risetimes`:

In [14]:
risetimes = [datetime.strptime(xpass['start'], "%Y-%m-%dT%H:%M:%S.%fZ") for xpass in passes]
risetimes

[datetime.datetime(2026, 3, 25, 1, 28, 0, 952000),
 datetime.datetime(2026, 3, 25, 13, 19, 50, 952000),
 datetime.datetime(2026, 3, 25, 14, 51, 45, 952000),
 datetime.datetime(2026, 3, 25, 16, 29, 5, 952000),
 datetime.datetime(2026, 3, 25, 18, 8, 40, 952000),
 datetime.datetime(2026, 3, 25, 19, 47, 40, 952000),
 datetime.datetime(2026, 3, 25, 21, 24, 10, 952000),
 datetime.datetime(2026, 3, 25, 23, 0, 40, 952000),
 datetime.datetime(2026, 3, 26, 0, 38, 55, 952000)]

Finally, use `risetime.strftime` to print these in a format that people understand:

```
e.g.
18/10/22 07:05
18/10/22 08:41
18/10/22 10:20
18/10/22 12:00
18/10/22 01:37
18/10/22 03:13
```



In [15]:
#ANSWER:
for risetime in risetimes:
    print(risetime.strftime('%d/%m/%dY %I:%M'))

25/03/25Y 01:28
25/03/25Y 01:19
25/03/25Y 02:51
25/03/25Y 04:29
25/03/25Y 06:08
25/03/25Y 07:47
25/03/25Y 09:24
25/03/25Y 11:00
26/03/26Y 12:38


Finally, here is an endpoint that tells us who is on board:

In [16]:
response = requests.get("http://api.open-notify.org/astros.json")

Referring to the methods used above, extract the number of astronauts and their names:

In [17]:
#ANSWER:
astros = response.json()
print(astros)
print(astros['number'])
for astronaut in astros['people']:
    print(astronaut['name'])

{'people': [{'craft': 'ISS', 'name': 'Oleg Kononenko'}, {'craft': 'ISS', 'name': 'Nikolai Chub'}, {'craft': 'ISS', 'name': 'Tracy Caldwell Dyson'}, {'craft': 'ISS', 'name': 'Matthew Dominick'}, {'craft': 'ISS', 'name': 'Michael Barratt'}, {'craft': 'ISS', 'name': 'Jeanette Epps'}, {'craft': 'ISS', 'name': 'Alexander Grebenkin'}, {'craft': 'ISS', 'name': 'Butch Wilmore'}, {'craft': 'ISS', 'name': 'Sunita Williams'}, {'craft': 'Tiangong', 'name': 'Li Guangsu'}, {'craft': 'Tiangong', 'name': 'Li Cong'}, {'craft': 'Tiangong', 'name': 'Ye Guangfu'}], 'number': 12, 'message': 'success'}
12
Oleg Kononenko
Nikolai Chub
Tracy Caldwell Dyson
Matthew Dominick
Michael Barratt
Jeanette Epps
Alexander Grebenkin
Butch Wilmore
Sunita Williams
Li Guangsu
Li Cong
Ye Guangfu


## HOMEWORK


1. Write a simple handler for the response status code (refer to lab resources slide for HTTP response codes). As this Jupyter Notebook is an interactive device, the handler does not need to manage subsequent code execution (i.e. by branching or aborting execution), although it should return something that could be used to do so if deployed in a Python program.

In [18]:
#ANSWER:
def handle_response_status(response):
    """
    Simple HTTP response handler.

    Parameters:
        response: requests.Response object

    Returns:
        bool: True if the request was successful, False otherwise
    """
    status = response.status_code

    if 200 <= status < 300:
        print(f"Success: {status}")
        return True

    elif status == 400:
        print("Bad Request: The server could not understand the request.")
        return False

    elif status == 401:
        print("Unauthorized: Authentication is required or failed.")
        return False

    elif status == 403:
        print("Forbidden: You do not have permission to access this resource.")
        return False

    elif status == 404:
        print("Not Found: The requested resource could not be found.")
        return False

    elif 500 <= status < 600:
        print(f"Server Error: {status}")
        return False

    else:
        print(f"Unhandled status code: {status}")
        return False

2. Test your response handler on some correct and incorrect API calls.

In [19]:
import requests

def handle_response_status(response):
    """
    Simple HTTP response handler.

    Parameters:
        response: requests.Response object

    Returns:
        bool: True if the request was successful, False otherwise
    """
    status = response.status_code

    if 200 <= status < 300:
        print(f"Success: {status}")
        return True

    elif status == 400:
        print("Bad Request: The server could not understand the request.")
        return False

    elif status == 401:
        print("Unauthorized: Authentication is required or failed.")
        return False

    elif status == 403:
        print("Forbidden: You do not have permission to access this resource.")
        return False

    elif status == 404:
        print("Not Found: The requested resource could not be found.")
        return False

    elif 500 <= status < 600:
        print(f"Server Error: {status}")
        return False

    else:
        print(f"Unhandled status code: {status}")
        return False


# Test 1: correct API call
response1 = requests.get("https://api.github.com")
print("Test 1:")
result1 = handle_response_status(response1)
print("Return value:", result1)
print()

# Test 2: incorrect endpoint -> 404
response2 = requests.get("https://api.github.com/this_endpoint_does_not_exist")
print("Test 2:")
result2 = handle_response_status(response2)
print("Return value:", result2)
print()

# Test 3: bad request example
response3 = requests.get("https://httpbin.org/status/400")
print("Test 3:")
result3 = handle_response_status(response3)
print("Return value:", result3)
print()

# Test 4: unauthorized example
response4 = requests.get("https://httpbin.org/status/401")
print("Test 4:")
result4 = handle_response_status(response4)
print("Return value:", result4)
print()

# Test 5: server error example
response5 = requests.get("https://httpbin.org/status/500")
print("Test 5:")
result5 = handle_response_status(response5)
print("Return value:", result5)

Test 1:
Success: 200
Return value: True

Test 2:
Not Found: The requested resource could not be found.
Return value: False

Test 3:
Bad Request: The server could not understand the request.
Return value: False

Test 4:
Unauthorized: Authentication is required or failed.
Return value: False

Test 5:
Server Error: 500
Return value: False


>

>

>



---



---



> > > > > > > > > © 2025 Institute of Data


---



---



